# Object-Oriented Programming

*4 Pillars · MRO · Protocols vs ABC · Dataclasses · classmethod/staticmethod · Descriptors · Real-World*


---
## Introduction


# Oop

*Run each cell with **Shift+Enter***

00 — Python Foundations: Object-Oriented Programming
====================================================

Runnable companion to PDF Chapter "P+ — Python Foundations" (OOP section).

Demonstrates the four pillars and Python's data model:
  * Encapsulation   — bundle state with the methods that guard it
  * Abstraction     — expose *what*, hide *how* (ABCs)
  * Inheritance     — reuse + specialize a base class
  * Polymorphism    — same call, different behavior
  * Dunder methods  — make your objects feel built-in
  * dataclasses     — auto-generate boilerplate

Run:  python oop.py


---
## 🧠 Notebook Mental Model: Object-Oriented Programming

> **Think of a class as a blueprint and an instance as the built object.**  
> OOP is about organising code around *data + the operations that guard it*, not around functions that act on bare data.

### The Four Pillars — Why Each Exists

```
ENCAPSULATION  ─── Bundle state + methods together; hide implementation details
                   WHY: prevents invalid state; allows change without breaking callers
                   HOW: private attributes (_name, __name mangling), @property
                   WHEN: any class that has invariants to protect

ABSTRACTION    ─── Expose WHAT, hide HOW
                   WHY: consumers don't need to know implementation; swap-in new types
                   HOW: ABCs (@abstractmethod); duck typing; Protocol
                   WHEN: designing pluggable interfaces, multiple implementations

INHERITANCE    ─── Reuse + specialise a parent class
                   WHY: avoid duplication; model IS-A relationships
                   HOW: class Child(Parent); super() to call parent methods
                   WHEN: genuine IS-A hierarchy; but prefer composition when in doubt

POLYMORPHISM   ─── Same call, different behaviour depending on runtime type
                   WHY: write generic code; add new types without changing callers
                   HOW: method overriding; duck typing (works without inheritance!)
                   WHEN: anytime you have `for item in items: item.process()`
```

### Key Decision Map

```
Need to reuse code?
  ├─ IS-A relationship  → Inheritance (`class Dog(Animal)`)
  └─ HAS-A relationship → Composition (`class Car: self.engine = Engine()`)
                          (Prefer composition over inheritance!)

Need to enforce an interface?
  ├─ For internal use    → Abstract Base Class (ABC + @abstractmethod)
  └─ For structural typing → Protocol (duck typing, no inheritance needed)

Need data storage + methods?
  ├─ Few fields, auto-boilerplate → @dataclass
  ├─ Named, immutable record      → NamedTuple / frozen dataclass
  └─ Full control                 → regular class
```

### MRO (Method Resolution Order) — Diamond Problem

```
class A: def go(self): return "A"
class B(A): def go(self): return "B"
class C(A): def go(self): return "C"
class D(B, C): pass    # Python uses C3 linearisation

D.__mro__ = [D, B, C, A, object]
D().go() → "B"   (first match in MRO)

Rule: always call super().method() in __init__ to chain correctly
```

### Why / What / How / When Summary

| Feature | WHY | WHEN |
|---------|-----|------|
| `@property` | Computed attribute with getter/setter validation | Protect state invariants |
| `@classmethod` | Alternative constructor; access class (not instance) | `from_string()`, `from_json()` |
| `@staticmethod` | Utility tied to class but needs no `self`/`cls` | Pure helper functions |
| `__slots__` | Disable `__dict__`; save ~50 % memory | Millions of small objects |
| Descriptor | Re-use attribute logic across multiple classes | Type validators, lazy loading |
| Dataclass | Auto-generate `__init__`, `__repr__`, `__eq__` | Simple data containers |


In [ ]:
from __future__ import annotations

from abc import ABC, abstractmethod
from dataclasses import dataclass, field

===========================================================================
ENCAPSULATION — hide internals behind methods that enforce invariants
===========================================================================


### 🧠 Mental Model: Encapsulation

**WHY** — An object without encapsulation is just a bag of data that anyone can corrupt. Encapsulation means the class *owns* its state and decides how it can change.

**WHAT** — Bundling data (attributes) with the methods that operate on that data, and restricting direct external access.

**HOW — Python's encapsulation conventions:**
```
self.name       — public: anyone can read/write
self._name      — protected by convention: "please don't touch"
self.__name     — name-mangled to self._ClassName__name: harder to access externally

@property       — controlled read access (getter)
@name.setter    — controlled write access with validation
@name.deleter   — controlled deletion
```

**WHEN:**
- Use `_prefix` to signal "internal implementation detail"
- Use `@property` when setting requires validation or computation
- Don't blindly add getters/setters for every attribute — only when the class has invariants to protect

**Key insight — what encapsulation protects:**
```python
class BankAccount:
    def __init__(self, balance=0):
        self._balance = balance    # protected: can only change via methods

    def deposit(self, amount):
        if amount <= 0:
            raise ValueError("deposit must be positive")  # INVARIANT enforced
        self._balance += amount

# Without encapsulation, anyone could do:
# account._balance = -9999999  ← no protection!
```


In [ ]:
class BankAccount:
    def __init__(self, balance: int = 0) -> None:
        self._balance = balance  # `_` = "private by convention"

    @property
    def balance(self) -> int:  # read-only view via @property
        return self._balance

    def deposit(self, amount: int) -> None:
        if amount <= 0:
            raise ValueError("deposit must be positive")
        self._balance += amount

    def withdraw(self, amount: int) -> None:
        if amount > self._balance:
            raise ValueError("insufficient funds")
        self._balance -= amount

===========================================================================
ABSTRACTION + INHERITANCE + POLYMORPHISM
===========================================================================


### 🧠 Mental Model: Abstraction, Inheritance & Polymorphism

**Abstraction — WHY/WHAT/HOW/WHEN:**
- **WHY**: Consumers need to know *what* to call, not *how* it's implemented — swap implementations without changing callers.
- **WHAT**: ABCs define a contract; `@abstractmethod` forces subclasses to implement it.
- **HOW**: If a method is not implemented, Python raises `TypeError` at instantiation — you can't create an incomplete object.
- **WHEN**: Multiple implementations of the same interface (payment processors, storage backends, serialisers).

**Inheritance — WHY/WHAT/HOW/WHEN:**
- **WHY**: Share behaviour across related types; model real IS-A hierarchies.
- **WHAT**: Child class inherits all attributes and methods from parent; can override or extend.
- **HOW**: `super()` calls the parent's method — chain `super().__init__()` in every `__init__`.
- **WHEN**: Genuine IS-A relationships. Prefer *composition* for HAS-A.

**Polymorphism — WHY/WHAT/HOW/WHEN:**
- **WHY**: Write generic code that works with any object implementing the right interface.
- **WHAT**: The same method call produces different behaviour based on the actual runtime type.
- **HOW**: Python uses *duck typing* — an object works if it has the right method, regardless of its type.
- **WHEN**: Collections of heterogeneous objects, plugin architectures, strategy pattern.

**Polymorphism diagram:**
```
animals = [Dog("Rex"), Cat("Luna"), Parrot("Polly")]
for a in animals:
    print(a.speak())   ← same call, different output (polymorphism)
```

**Composition vs Inheritance decision:**
```
IS-A: "A Dog IS-A Animal"          → Inheritance
HAS-A: "A Car HAS-A Engine"       → Composition (store as attribute)
CAN-DO: "A Duck CAN-DO Quackable" → Protocol / duck typing
```


In [ ]:
class Animal(ABC):
    def __init__(self, name: str) -> None:
        self.name = name

    @abstractmethod
    def speak(self) -> str:  # subclasses MUST implement — the abstraction
        ...

    def introduce(self) -> str:
        return f"{self.name} says {self.speak()}"


class Dog(Animal):
    def speak(self) -> str:  # overrides base -> polymorphism
        return "Woof"


class Cat(Animal):
    def speak(self) -> str:
        return "Meow"

===========================================================================
DUNDER (MAGIC) METHODS + OPERATOR OVERLOADING
===========================================================================


### 🧠 Mental Model: Dunder (Magic) Methods

**WHY** — Dunder methods let your custom class plug into Python's built-in syntax and protocols. They're how Python achieves uniform behaviour across all types.

**WHAT** — Dunder = "double underscore". Methods like `__len__`, `__add__`, `__iter__` are called automatically by Python's built-in operators and functions.

**HOW — the lookup mechanism:**
```
len(obj)       → obj.__len__()
obj + other    → obj.__add__(other)   (fallback: other.__radd__(obj))
str(obj)       → obj.__str__()
repr(obj)      → obj.__repr__()
obj[key]       → obj.__getitem__(key)
key in obj     → obj.__contains__(key)  (or iterate via __iter__ if not defined)
for x in obj   → iter(obj) → obj.__iter__() → repeated __next__()
with obj       → obj.__enter__() / obj.__exit__()
```

**The most important dunders to know:**

| Dunder | Called by | Purpose |
|--------|-----------|---------|
| `__init__` | `MyClass()` | Constructor / initialiser |
| `__repr__` | `repr(x)`, debugger | Unambiguous developer string |
| `__str__` | `str(x)`, `print(x)` | Human-readable string |
| `__eq__` | `x == y` | Value equality |
| `__hash__` | `hash(x)`, `dict[x]`, `{x}` | Must be defined if `__eq__` is |
| `__lt__` | `x < y`, `sorted()` | Use with `@total_ordering` |
| `__len__` | `len(x)`, truthiness | Container size |
| `__getitem__` | `x[key]` | Indexing and slicing |
| `__iter__` | `for x in obj` | Makes class iterable |
| `__enter__`/`__exit__` | `with x:` | Context manager protocol |
| `__call__` | `x()` | Makes instances callable |
| `__add__`, `__mul__`, … | `x + y`, `x * y` | Operator overloading |

**The `__eq__`/`__hash__` contract (crucial!):**
```
If a == b, then hash(a) MUST == hash(b)
If you define __eq__, Python sets __hash__ = None (unhashable)
  → You MUST also define __hash__ to use instances in dicts/sets

@dataclass(frozen=True) handles this automatically.
```


In [ ]:
@dataclass  # auto-generates __init__, __repr__, __eq__
class Point:
    x: int
    y: int

    def __add__(self, other: Point) -> Point:  # enables Point + Point
        return Point(self.x + other.x, self.y + other.y)

    def __len__(self) -> int:  # enables len(point) -> manhattan distance
        return abs(self.x) + abs(self.y)


@dataclass
class Stack:
    """Shows __len__, __getitem__, __iter__, __bool__ making a class native."""

    _items: list[int] = field(default_factory=list)

    def push(self, v: int) -> None:
        self._items.append(v)

    def __len__(self) -> int:
        return len(self._items)

    def __getitem__(self, i: int) -> int:
        return self._items[i]

    def __bool__(self) -> bool:
        return bool(self._items)

classmethod / staticmethod


### 🧠 Mental Model: `@classmethod` vs `@staticmethod` vs Instance Method

**WHY** — Sometimes functionality is *related* to a class but doesn't need an instance. Choosing the right binding keeps APIs clean and communicates intent.

| | Instance method | `@classmethod` | `@staticmethod` |
|-|----------------|----------------|-----------------|
| **First arg** | `self` (instance) | `cls` (the class) | None |
| **Can access** | Instance + class state | Class state only | Neither |
| **Inherited as?** | Yes | Yes (cls = subclass!) | Yes |
| **Called on** | instance or class | class or instance | class or instance |

**WHEN to use each:**

```python
class Date:
    def __init__(self, year, month, day):
        self.year, self.month, self.day = year, month, day

    # Instance method — needs specific date
    def is_weekend(self) -> bool:
        import datetime
        return datetime.date(self.year, self.month, self.day).weekday() >= 5

    # classmethod — ALTERNATIVE CONSTRUCTOR (most common use)
    @classmethod
    def from_string(cls, s: str) -> "Date":
        y, m, d = map(int, s.split("-"))
        return cls(y, m, d)    # cls = Date or subclass; handles inheritance!

    # staticmethod — UTILITY related to the class, needs no state
    @staticmethod
    def is_valid(year: int, month: int, day: int) -> bool:
        return 1 <= month <= 12 and 1 <= day <= 31
```

**Key insight — `@classmethod` with inheritance:**
```python
class SpecialDate(Date):
    pass

# @classmethod: cls = SpecialDate → returns SpecialDate instance
d = SpecialDate.from_string("2024-01-15")
type(d)  # SpecialDate, not Date!

# If from_string used Date(...) instead of cls(...), it would break inheritance
```


In [ ]:
class Temperature:
    def __init__(self, celsius: float) -> None:
        self.celsius = celsius

    @classmethod
    def from_fahrenheit(cls, f: float) -> Temperature:  # alternative constructor
        return cls((f - 32) * 5 / 9)

    @staticmethod
    def is_freezing(celsius: float) -> bool:  # utility, no self/cls needed
        return celsius <= 0


def main() -> None:
    print("=" * 68)
    print("PYTHON FOUNDATIONS — oop.py")
    print("=" * 68)

    acct = BankAccount(100)
    acct.deposit(50)
    acct.withdraw(30)
    assert acct.balance == 120
    try:
        acct.withdraw(10_000)
    except ValueError as e:
        assert "insufficient" in str(e)
    print("encapsulation:", acct.balance, "(guarded by deposit/withdraw)")

    animals: list[Animal] = [Dog("Rex"), Cat("Mia")]
    speeches = [a.introduce() for a in animals]  # same call, different behavior
    assert speeches == ["Rex says Woof", "Mia says Meow"]
    print("polymorphism:", speeches)

    p = Point(1, 2) + Point(3, 4)
    assert p == Point(4, 6) and len(p) == 10
    print("dunder:", p, "| len(p) =", len(p))

    s = Stack()
    s.push(1)
    s.push(2)
    assert len(s) == 2 and s[0] == 1 and bool(s) is True
    print("dunder container: len/getitem/bool work on Stack")

    t = Temperature.from_fahrenheit(32)
    assert abs(t.celsius) < 1e-9 and Temperature.is_freezing(t.celsius)
    print("class/staticmethod: 32F ->", round(t.celsius, 2), "C (freezing)")

    print("-" * 68)
    print("All OOP demos passed ✔")


if __name__ == "__main__":
    # Keep Unicode output safe even when stdout is redirected/piped (Windows cp1252 fallback).
    import sys
    if hasattr(sys.stdout, "reconfigure"):
        sys.stdout.reconfigure(encoding="utf-8")
    main()

---
## OOP — Complete + MRO + DI


# Oop Complete

*Run each cell with **Shift+Enter***

Python OOP — Complete Reference: All Pillars, Patterns & Internals
===================================================================
Mental model: OOP is about DRAWING BOUNDARIES — bundling the data that
  changes together with the operations that change it.

PART 1 : Encapsulation       — @property, name mangling, slots
PART 2 : Inheritance         — single, multi, MRO, super(), mixins
PART 3 : Abstraction         — ABC, @abstractmethod, Protocol
PART 4 : Polymorphism        — duck typing, isinstance, double dispatch
PART 5 : Dunder methods      — ALL major protocols (repr/eq/hash/iter/…)
PART 6 : class/staticmethod  — alternative constructors, utilities
PART 7 : dataclasses         — @dataclass, frozen, slots, __post_init__
PART 8 : Descriptors         — non-data, data, __get__/__set__/__set_name__
PART 9 : __slots__           — memory optimisation, attribute restriction
PART 10: __init_subclass__   — plugin registry without metaclass
PART 11: Metaclasses         — type, __new__, __prepare__, custom metaclass
PART 12: Design patterns     — singleton, factory, observer, mixin

Run: python oop_complete.py

In [ ]:
from __future__ import annotations

import sys
import weakref
from abc import ABC, abstractmethod
from collections import defaultdict
from dataclasses import dataclass, field, fields
from functools import total_ordering
from typing import Any, ClassVar, Protocol

## PART 1 — ENCAPSULATION
Mental model: hide the HOW behind a clean interface that expresses the WHAT.
  Three levels of "privacy" in Python (all enforced by convention, not the VM):
    name       → public (no restriction)
    _name      → "protected" convention — internal use, not in public API
    __name     → name-mangled to _ClassName__name (avoid accidental override)

In [ ]:
class BankAccount:
    """
    Demonstrates encapsulation:
    - _balance is "private" by convention (single underscore)
    - @property exposes a READ-ONLY view (getter only)
    - deposit/withdraw enforce invariants (negative balance impossible)
    - __audit is name-mangled to prevent accidental override in subclasses
    """
    def __init__(self, owner: str, initial: float = 0) -> None:
        self._balance: float = initial
        self._owner   = owner
        self.__audit: list[str] = []          # _BankAccount__audit

    # @property: computed attribute — called on READ, no set allowed
    @property
    def balance(self) -> float:
        return self._balance

    @property
    def owner(self) -> str:
        return self._owner

    # @balance.setter: validate on WRITE
    @owner.setter
    def owner(self, name: str) -> None:
        if not name.strip():
            raise ValueError("owner name cannot be blank")
        self._owner = name

    def deposit(self, amount: float) -> None:
        if amount <= 0:
            raise ValueError("deposit must be positive")
        self._balance += amount
        self.__audit.append(f"+{amount}")

    def withdraw(self, amount: float) -> None:
        if amount > self._balance:
            raise ValueError("insufficient funds")
        self._balance -= amount
        self.__audit.append(f"-{amount}")

    def audit_log(self) -> list[str]:
        return list(self.__audit)   # return copy — caller cannot mutate internals


def demo_encapsulation() -> None:
    acct = BankAccount("Ada", 100)
    acct.deposit(50)
    acct.withdraw(30)
    assert acct.balance == 120
    assert acct.audit_log() == ["+50", "-30"]
    try:
        acct.withdraw(10_000)
    except ValueError as e:
        assert "insufficient" in str(e)

    # Name mangling: __audit is stored as _BankAccount__audit
    assert "_BankAccount__audit" in dir(acct)

    # @property setter
    acct.owner = "Bob"
    assert acct.owner == "Bob"
    try:
        acct.owner = "  "
    except ValueError:
        pass

    print("Part 1 (Encapsulation): ✓")

## PART 2 — INHERITANCE
Mental model: inheritance means "IS-A". Use it ONLY for true IS-A relationships.
  Prefer composition for HAS-A (a Car HAS-A Engine, not IS-A Engine).

In [ ]:
class Vehicle:
    """Base class — common interface for all vehicles."""
    def __init__(self, make: str, model: str, year: int) -> None:
        self.make, self.model, self.year = make, model, year

    def describe(self) -> str:
        return f"{self.year} {self.make} {self.model}"

    def start(self) -> str:
        return "vroom"


class Car(Vehicle):
    """Single inheritance."""
    def __init__(self, make: str, model: str, year: int, doors: int = 4) -> None:
        super().__init__(make, model, year)   # ALWAYS call super().__init__
        self.doors = doors

    def start(self) -> str:                  # override
        return f"Car '{self.model}' starting: {super().start()}"


class ElectricMixin:
    """
    Mixin — adds capabilities without IS-A semantics.
    No __init__ parameters, designed for multiple inheritance.
    """
    battery_kwh: float = 75.0

    def start(self) -> str:
        return "silent electric hum"

    def charge_status(self) -> str:
        return f"{self.battery_kwh} kWh"


class ElectricCar(ElectricMixin, Car):
    """
    Multiple inheritance: ElectricCar IS-A Car that also HAS electric capability.
    MRO: ElectricCar → ElectricMixin → Car → Vehicle → object
    super().start() follows the MRO — calls ElectricMixin.start()
    """
    pass


def demo_inheritance() -> None:
    car = Car("Toyota","Camry", 2023)
    assert "Camry" in car.start()

    ev = ElectricCar("Tesla","Model 3", 2024)
    assert ev.start() == "silent electric hum"   # ElectricMixin wins in MRO
    assert ev.describe() == "2024 Tesla Model 3" # from Vehicle via Car

    # MRO
    mro_names = [c.__name__ for c in ElectricCar.__mro__]
    assert mro_names == ["ElectricCar","ElectricMixin","Car","Vehicle","object"]

    # isinstance checks full MRO
    assert isinstance(ev, Car) and isinstance(ev, Vehicle)
    assert isinstance(ev, ElectricMixin)

    print("Part 2 (Inheritance): ✓")

## PART 2B — MRO & MULTIPLE INHERITANCE — FULL NUANCES

Mental model: Python must answer "which class provides this method?" when
  multiple base classes all define it.  The answer is always deterministic
  because of C3 linearization.  Learn the rules once; every surprise
  disappears.

Topics covered:
  2B-1  How C3 linearization works (the algorithm, by hand)
  2B-2  The diamond problem — and why Python solves it correctly
  2B-3  Cooperative super() — ALL classes must cooperate
  2B-4  Mixin ordering matters (D(B,C) ≠ D(C,B))
  2B-5  MRO failure — when no consistent linearization exists (TypeError)
  2B-6  super() with different __init__ signatures (**kwargs forwarding)
  2B-7  super() for property / classmethod / staticmethod
  2B-8  Runtime MRO inspection tools

`─── 2B-1 : C3 LINEARIZATION ─────────────────────────────────────────────────`
The C3 algorithm computes an MRO from a class and its bases by merging their
linearizations while respecting LOCAL PRECEDENCE ORDER (left-to-right bases)
and the MONOTONICITY constraint (a class always appears before its parents).

Rule (informal): take the head of the first non-empty list whose head does
NOT appear in the tail of any other list; add it to the result; remove it
from all lists; repeat.

Example: class D(B, C)  where B(A), C(A), A(object)

  L[D] = D + merge( L[B],  L[C],  [B,C] )
         = D + merge( [B,A,obj], [C,A,obj], [B,C] )
  head [B,A,obj] = B — not in tails [A,obj],[A,obj],[C] → take B
         = D + B + merge( [A,obj], [C,A,obj], [C] )
  head [A,obj]   = A — IS in tail [C,A,obj] → skip
  head [C,A,obj] = C — not in any tail → take C
         = D + B + C + merge( [A,obj], [A,obj], [] )
  head [A,obj]   = A → take A
         = D + B + C + A + merge( [obj], [obj] )
         = D + B + C + A + object

  Final MRO:  D → B → C → A → object

In [ ]:
class A_c3:
    def method(self) -> str: return "A"

class B_c3(A_c3):
    def method(self) -> str: return "B→" + super().method()

class C_c3(A_c3):
    def method(self) -> str: return "C→" + super().method()

class D_c3(B_c3, C_c3):
    """Diamond: D(B,C), B(A), C(A), A(object).  MRO: D→B→C→A→object."""
    pass

`─── 2B-2 : THE DIAMOND PROBLEM ──────────────────────────────────────────────`
Without MRO: calling A.method() twice (once via B, once via C) would
duplicate side-effects and risk double-init bugs.
With C3 + cooperative super(): A.method() is called EXACTLY ONCE.

In [ ]:
class Logger_d:
    """Root of the diamond — should run exactly once."""
    calls: list[str] = []

    def setup(self) -> None:
        Logger_d.calls.append("Logger_d.setup")
        # super().setup() here would call object.setup — absent, so we stop
        # BUT in a cooperative chain every class MUST call super() even if
        # it thinks it's the top, so the chain doesn't break mid-hierarchy.


class FileLogger(Logger_d):
    def setup(self) -> None:
        FileLogger._local = True
        super().setup()                    # forwards to Logger_d via MRO
        Logger_d.calls.append("FileLogger.setup")


class NetworkLogger(Logger_d):
    def setup(self) -> None:
        super().setup()                    # forwards to Logger_d via MRO
        Logger_d.calls.append("NetworkLogger.setup")


class HybridLogger(FileLogger, NetworkLogger):
    """
    MRO: HybridLogger → FileLogger → NetworkLogger → Logger_d → object
    setup() call chain (each calls super()):
      HybridLogger.setup()   → (no override, so)
      FileLogger.setup()     → super() → NetworkLogger.setup()
                                          → super() → Logger_d.setup()
    Logger_d.setup() runs ONCE despite two paths to it.
    """
    pass

`─── 2B-3 : COOPERATIVE super() — EVERY CLASS MUST COOPERATE ─────────────────`
If any class in the chain does NOT call super(), the chain is broken.
The class that breaks it silently swallows all parent calls.

In [ ]:
class Base_coop:
    def greet(self) -> str:
        return "Base"

class GreetA(Base_coop):
    def greet(self) -> str:
        return "A+" + super().greet()      # ✓ cooperative

class GreetB(Base_coop):
    def greet(self) -> str:
        return "B+" + super().greet()      # ✓ cooperative

class GreetBroken(Base_coop):
    def greet(self) -> str:
        return "BROKEN"                    # ✗ does NOT call super()
        # Any class after GreetBroken in the MRO is silently skipped!

class GoodCombo(GreetA, GreetB):
    """MRO: GoodCombo→GreetA→GreetB→Base_coop→object.  All cooperate."""
    pass

class BrokenCombo(GreetA, GreetBroken):
    """
    MRO: BrokenCombo→GreetA→GreetBroken→Base_coop→object.
    GreetBroken doesn't call super() → GreetA's super() hits GreetBroken
    which returns immediately → Base_coop.greet() is NEVER called.
    """
    pass

`─── 2B-4 : MIXIN ORDERING MATTERS ───────────────────────────────────────────`
The order you list bases in class X(B, C) determines who wins.
Left-most class has HIGHEST priority in the MRO.

In [ ]:
class JSONMixin_o:
    def serialize(self) -> str: return "json"

class XMLMixin_o:
    def serialize(self) -> str: return "xml"

class ModelBase_o:
    def serialize(self) -> str: return "base"

class JsonFirst(JSONMixin_o, XMLMixin_o, ModelBase_o):
    """MRO: JsonFirst → JSONMixin_o → XMLMixin_o → ModelBase_o → object"""
    pass

class XmlFirst(XMLMixin_o, JSONMixin_o, ModelBase_o):
    """MRO: XmlFirst → XMLMixin_o → JSONMixin_o → ModelBase_o → object"""
    pass

`─── 2B-5 : MRO FAILURE — INCONSISTENT HIERARCHY ─────────────────────────────`
Python raises TypeError at class-definition time when no valid C3 ordering
can be found.  This protects you from ambiguous hierarchies at import, not
at runtime.

Classic failure:
  class X(A, B): pass    where A(B) already exists
  → X wants A before B, but A already lists B after A.
  The merge algorithm gets stuck; Python raises TypeError.

In [ ]:
def demonstrate_mro_failure() -> None:
    class P: pass
    class Q(P): pass

    try:
        # X(Q, P) is fine: Q→P is consistent with Q(P)
        class Fine(Q, P): pass         # MRO: Fine→Q→P→object  ✓

        # X(P, Q) is NOT fine: X wants P before Q, but Q is a subclass of P,
        # so P must come AFTER Q.  The C3 merge has no valid solution.
        class Broken(P, Q): pass       # TypeError: cannot create consistent MRO ✗
        raise AssertionError("should have raised TypeError")
    except TypeError as e:
        assert "consistent" in str(e).lower() or "mro" in str(e).lower()

`─── 2B-6 : super() WITH DIFFERENT __init__ SIGNATURES ───────────────────────`
The **kwargs forwarding pattern: every __init__ absorbs what it needs and
passes the rest up the chain via **kwargs.  This keeps cooperative __init__
working even when classes add their own parameters.

In [ ]:
class Person:
    def __init__(self, name: str, **kwargs: Any) -> None:
        super().__init__(**kwargs)         # passes remaining kwargs up
        self.name = name


class Employee(Person):
    def __init__(self, employee_id: int, **kwargs: Any) -> None:
        super().__init__(**kwargs)         # passes remaining kwargs up
        self.employee_id = employee_id


class Manager(Person):
    def __init__(self, department: str, **kwargs: Any) -> None:
        super().__init__(**kwargs)
        self.department = department


class SeniorManager(Employee, Manager):
    """
    MRO: SeniorManager → Employee → Manager → Person → object
    All three classes use **kwargs, so a single constructor call works.
    """
    def __init__(self, **kwargs: Any) -> None:
        super().__init__(**kwargs)

`─── 2B-7 : super() WITH property / classmethod / staticmethod ───────────────`

In [ ]:
class Base_prop:
    @property
    def value(self) -> int:
        return 10

    @classmethod
    def create(cls) -> "Base_prop":
        return cls()

    @staticmethod
    def helper() -> str:
        return "base-helper"


class Child_prop(Base_prop):
    @property
    def value(self) -> int:
        return super().value * 2          # delegate to parent property

    @classmethod
    def create(cls) -> "Child_prop":
        instance = super().create()       # polymorphic: returns Child_prop
        return instance

    @staticmethod
    def helper() -> str:
        return Base_prop.helper() + "+child"  # staticmethod: no super(), call directly

`─── 2B-8 : RUNTIME MRO INSPECTION ──────────────────────────────────────────`
Three equivalent ways to inspect the MRO at runtime:
  Cls.__mro__   — tuple of classes in resolution order
  Cls.mro()     — list (same content, but a fresh list each call)
  inspect.getmro(Cls) — same as __mro__ but from the inspect module

In [ ]:
def demo_mro() -> None:
    # ── 2B-1: C3 linearization result ────────────────────────────────────────
    mro = [c.__name__ for c in D_c3.__mro__]
    assert mro == ["D_c3","B_c3","C_c3","A_c3","object"]

    # Cooperative chain: each super() call passes through the MRO
    assert D_c3().method() == "B→C→A"    # A runs once even in diamond

    # ── 2B-2: diamond — Logger_d.setup() runs exactly once ───────────────────
    Logger_d.calls.clear()
    HybridLogger().setup()
    # FileLogger.setup → NetworkLogger.setup → Logger_d.setup (once)
    assert Logger_d.calls.count("Logger_d.setup") == 1
    assert "FileLogger.setup" in Logger_d.calls
    assert "NetworkLogger.setup" in Logger_d.calls

    # ── 2B-3: cooperative vs broken ──────────────────────────────────────────
    assert GoodCombo().greet()   == "A+B+Base"   # full chain
    assert BrokenCombo().greet() == "A+BROKEN"   # Base_coop skipped silently!

    # ── 2B-4: ordering changes the winner ────────────────────────────────────
    assert JsonFirst().serialize() == "json"      # JSONMixin_o wins (listed first)
    assert XmlFirst().serialize()  == "xml"       # XMLMixin_o wins (listed first)
    assert [c.__name__ for c in JsonFirst.__mro__][:3] == ["JsonFirst","JSONMixin_o","XMLMixin_o"]
    assert [c.__name__ for c in XmlFirst.__mro__][:3]  == ["XmlFirst","XMLMixin_o","JSONMixin_o"]

    # ── 2B-5: MRO failure detected at class-definition time ──────────────────
    demonstrate_mro_failure()

    # ── 2B-6: cooperative __init__ with different signatures ─────────────────
    sm = SeniorManager(name="Alice", employee_id=42, department="Engineering")
    assert sm.name        == "Alice"
    assert sm.employee_id == 42
    assert sm.department  == "Engineering"
    # Each class only consumed its own kwarg; the rest flowed up the chain.

    # ── 2B-7: property / classmethod via super() ─────────────────────────────
    c = Child_prop()
    assert c.value == 20                          # super().value * 2
    created = Child_prop.create()
    assert type(created) is Child_prop            # classmethod is polymorphic
    assert Child_prop.helper() == "base-helper+child"

    # ── 2B-8: runtime inspection ─────────────────────────────────────────────
    import inspect
    assert HybridLogger.__mro__ == tuple(HybridLogger.mro())
    assert inspect.getmro(HybridLogger) == HybridLogger.__mro__
    # __mro__ vs mro(): __mro__ is a cached tuple; mro() computes a fresh list
    assert type(HybridLogger.__mro__) is tuple
    assert type(HybridLogger.mro())   is list

    # super() with explicit arguments (needed inside classmethods / staticmethods
    # where the zero-arg form can't infer context):
    class Explicit(Base_prop):
        @classmethod
        def create(cls):
            return super(Explicit, cls).create()  # explicit super()
    assert type(Explicit.create()) is Explicit

    print("Part 2B (MRO & multiple inheritance nuances): ✓")

> 💡 **Mental model: expose WHAT the interface does, hide HOW it does it.**
  ABC  (Abstract Base Class) — explicit, nominal, can share implementation.
  Protocol — structural, duck-typed, zero coupling (no import needed).

In [ ]:
class Shape(ABC):
    """
    Abstract base class — cannot be instantiated directly.
    @abstractmethod forces subclasses to implement area() and perimeter().
    """
    @abstractmethod
    def area(self) -> float: ...

    @abstractmethod
    def perimeter(self) -> float: ...

    def describe(self) -> str:        # concrete method — shared by all subclasses
        return f"{type(self).__name__}: area={self.area():.2f}, perimeter={self.perimeter():.2f}"


class Circle(Shape):
    def __init__(self, r: float) -> None: self.r = r
    def area(self)      -> float: return 3.14159 * self.r ** 2
    def perimeter(self) -> float: return 2 * 3.14159 * self.r


class Rectangle(Shape):
    def __init__(self, w: float, h: float) -> None: self.w, self.h = w, h
    def area(self)      -> float: return self.w * self.h
    def perimeter(self) -> float: return 2 * (self.w + self.h)

Protocol — structural (duck) typing; implementer needs NO knowledge of the Protocol

In [ ]:
class SupportsArea(Protocol):
    def area(self) -> float: ...

def total_area(shapes: list[SupportsArea]) -> float:
    return sum(s.area() for s in shapes)  # works with ANY class with .area()


class Triangle:
    """NOT a Shape subclass — yet still accepted by total_area() via Protocol."""
    def __init__(self, b: float, h: float) -> None: self.b, self.h = b, h
    def area(self) -> float: return 0.5 * self.b * self.h


def demo_abstraction() -> None:
    try:
        Shape()                          # TypeError — abstract
    except TypeError:
        pass

    shapes = [Circle(5), Rectangle(3,4), Triangle(6,4)]
    approx = total_area(shapes)
    assert abs(approx - (78.54 + 12 + 12)) < 0.1

    # ABC provides concrete behaviour too
    print(Circle(3).describe())

    print("Part 3 (Abstraction/Protocol): ✓")

## PART 4 — POLYMORPHISM
Mental model: "many forms" — the SAME method name, different behaviour.
  In Python, polymorphism is achieved via duck typing:
  "if it has .area() I'll call .area(); I don't care what TYPE it is".
  No need for interfaces — any object with the right methods qualifies.

In [ ]:
class Dog:
    def speak(self) -> str: return "Woof!"

class Cat:
    def speak(self) -> str: return "Meow!"

class Duck:
    def speak(self) -> str: return "Quack!"

def make_noise(animals: list) -> list[str]:
    """Duck typing: ANY object with .speak() works. No shared base class needed."""
    return [a.speak() for a in animals]


def demo_polymorphism() -> None:
    animals = [Dog(), Cat(), Duck()]
    assert make_noise(animals) == ["Woof!", "Meow!", "Quack!"]

    # isinstance / issubclass for safe runtime type checking
    assert isinstance(Circle(1), Shape)        # also checks all base classes
    assert issubclass(Circle, Shape)
    assert not isinstance(Triangle(1,1), Shape)  # Triangle is NOT an ABC subclass

    print("Part 4 (Polymorphism): ✓")

## PART 5 — DUNDER (MAGIC) METHODS — THE COMPLETE DATA MODEL
Mental model: dunder methods are hooks into Python's operator and syntax layer.
  Implement them and your class behaves like a native Python type.

In [ ]:
@total_ordering
class Money:
    """
    A well-behaved immutable value object demonstrating the complete dunder set.
    Rule: __eq__ and __hash__ MUST be consistent (equal ⇒ same hash).
    """
    __slots__ = ("_cents",)          # memory optimisation + immutability enforcement

    def __init__(self, dollars: float):
        object.__setattr__(self, "_cents", round(dollars * 100))

    # ── String representation ───────────────────────────────────────────────
    def __repr__(self) -> str:       # eval()-able; for developers
        return f"Money({self._cents / 100:.2f})"

    def __str__(self) -> str:        # human-readable; for users
        return f"${self._cents / 100:,.2f}"

    def __format__(self, spec: str) -> str:
        return format(str(self), spec)

    # ── Value equality & hashing ─────────────────────────────────────────────
    def __eq__(self, other: object) -> bool:
        return isinstance(other, Money) and self._cents == other._cents

    def __hash__(self) -> int:
        return hash(self._cents)     # equal objects MUST hash equal

    # ── Ordering (total_ordering derives <=, >, >= from __lt__ + __eq__) ────
    def __lt__(self, other: "Money") -> bool:
        return self._cents < other._cents

    # ── Arithmetic ───────────────────────────────────────────────────────────
    def __add__(self, other: "Money") -> "Money":
        return Money((self._cents + other._cents) / 100)

    def __sub__(self, other: "Money") -> "Money":
        return Money((self._cents - other._cents) / 100)

    def __mul__(self, factor: float) -> "Money":
        return Money(self._cents * factor / 100)

    def __truediv__(self, divisor: float) -> "Money":
        return Money(self._cents / (divisor * 100))

    def __neg__(self) -> "Money":
        return Money(-self._cents / 100)

    def __abs__(self) -> "Money":
        return Money(abs(self._cents) / 100)

    # ── Immutability guard ───────────────────────────────────────────────────
    def __setattr__(self, name: str, val: Any) -> None:
        raise AttributeError("Money is immutable")

    def __bool__(self) -> bool:
        return self._cents != 0


class Playlist:
    """Container protocol: __len__, __getitem__, __setitem__, __contains__, __iter__."""
    def __init__(self, songs: list[str]) -> None:
        self._songs = list(songs)

    def __len__(self)              -> int:         return len(self._songs)
    def __getitem__(self, i)       -> str:         return self._songs[i]
    def __setitem__(self, i, val)  -> None:        self._songs[i] = val
    def __delitem__(self, i)       -> None:        del self._songs[i]
    def __contains__(self, song)   -> bool:        return song in self._songs
    def __iter__(self):                            return iter(self._songs)
    def __reversed__(self):                        return reversed(self._songs)
    def __repr__(self)             -> str:         return f"Playlist({self._songs})"

    # Make Playlist behave like a list — supports + and *
    def __add__(self, other: "Playlist") -> "Playlist":
        return Playlist(self._songs + other._songs)


class Logger:
    """Context manager protocol: __enter__ / __exit__."""
    def __init__(self, name: str) -> None:
        self.name, self.log = name, []

    def __enter__(self) -> "Logger":
        self.log.append("BEGIN"); return self

    def __exit__(self, exc_type, exc, tb) -> bool:
        self.log.append("ERROR" if exc_type else "END")
        return False    # DO NOT suppress exceptions (return True would swallow them)


class Multiplier:
    """__call__: make an instance callable like a function."""
    def __init__(self, factor: float): self.factor = factor
    def __call__(self, x: float) -> float: return x * self.factor


class Config:
    """__getattr__ / __setattr__ / __delattr__: intercept ALL attribute access."""
    def __init__(self, data: dict) -> None:
        object.__setattr__(self, "_data", data)

    def __getattr__(self, name: str) -> Any:
        # only called when NORMAL lookup fails (not in instance dict or class)
        try:   return self._data[name]
        except KeyError: raise AttributeError(name) from None

    def __setattr__(self, name: str, val: Any) -> None:
        if name.startswith("_"):
            object.__setattr__(self, name, val)
        else:
            self._data[name] = val

    def __delattr__(self, name: str) -> None:
        del self._data[name]

    def __dir__(self) -> list[str]:
        return list(super().__dir__()) + list(self._data)


class CountingDict(dict):
    """__missing__: called when key not found in dict subclass."""
    def __missing__(self, key: Any) -> int:
        self[key] = 0   # auto-create with 0
        return 0


def demo_dunders() -> None:
    # Money value object
    m1, m2 = Money(10.50), Money(5.25)
    assert m1 + m2 == Money(15.75)
    assert m1 > m2
    assert repr(m1) == "Money(10.50)"
    assert str(m1) == "$10.50"
    assert {m1, Money(10.50)} == {m1}   # hashable
    try:
        m1._cents = 0                  # __setattr__ blocks mutation
    except AttributeError:
        pass
    assert sorted([Money(3), Money(1), Money(2)]) == [Money(1), Money(2), Money(3)]

    # Playlist container
    pl = Playlist(["a","b","c"])
    assert len(pl) == 3 and pl[0] == "a"
    assert "b" in pl
    assert list(pl) == ["a","b","c"]
    pl[1] = "B"
    assert "B" in pl
    combined = pl + Playlist(["d","e"])
    assert len(combined) == 5

    # Logger context manager
    log = []
    with Logger("tx") as l:
        log = l.log; l.log.append("work")
    assert log == ["BEGIN","work","END"]
    try:
        with Logger("tx2") as l2:
            l2.log.append("work")
            raise ValueError("boom")
    except ValueError:
        pass
    assert l2.log == ["BEGIN","work","ERROR"]   # cleanup ran despite exception

    # Callable
    triple = Multiplier(3)
    assert triple(7) == 21 and callable(triple)

    # Config __getattr__
    cfg = Config({"host":"localhost","port":5432})
    assert cfg.host == "localhost"
    cfg.db = "mydb"                    # __setattr__
    assert cfg._data["db"] == "mydb"

    # __missing__
    cd = CountingDict()
    cd["x"] += 1; cd["x"] += 1
    assert cd["x"] == 2
    assert cd["new_key"] == 0          # no KeyError

    print("Part 5 (Dunders): ✓")

## PART 6 — classmethod & staticmethod
Mental model:
  classmethod  — receives `cls` (the class, not the instance).
                 Use for ALTERNATIVE CONSTRUCTORS (factory pattern).
  staticmethod — receives nothing (no self, no cls).
                 Use for UTILITIES that logically belong in the namespace.

In [ ]:
class Temperature:
    def __init__(self, celsius: float) -> None:
        self.celsius = celsius

    @classmethod
    def from_fahrenheit(cls, f: float) -> "Temperature":
        """Alternative constructor — polymorphic: returns the class it's called on."""
        return cls((f - 32) * 5 / 9)

    @classmethod
    def from_kelvin(cls, k: float) -> "Temperature":
        return cls(k - 273.15)

    @staticmethod
    def is_freezing(celsius: float) -> bool:
        """Pure utility — doesn't need self or cls."""
        return celsius <= 0

    def __repr__(self) -> str:
        return f"Temperature({self.celsius:.2f}°C)"

    def to_fahrenheit(self) -> float:
        return self.celsius * 9 / 5 + 32

    def to_kelvin(self) -> float:
        return self.celsius + 273.15


class AbsoluteTemperature(Temperature):
    """classmethod is polymorphic — from_fahrenheit returns AbsoluteTemperature."""
    pass


def demo_class_static() -> None:
    t1 = Temperature.from_fahrenheit(212)
    assert abs(t1.celsius - 100) < 0.01

    t2 = Temperature.from_kelvin(0)
    assert abs(t2.celsius - (-273.15)) < 0.01

    assert Temperature.is_freezing(0)
    assert not Temperature.is_freezing(20)

    # classmethod is polymorphic
    abs_t = AbsoluteTemperature.from_fahrenheit(32)
    assert isinstance(abs_t, AbsoluteTemperature)   # returns subclass

    print("Part 6 (class/staticmethod): ✓")

## PART 7 — DATACLASSES
Mental model: auto-generate boilerplate (__init__, __repr__, __eq__, etc.)
  so you focus on WHAT the data IS, not HOW it's stored.
  Options: frozen (immutable), slots (efficient), order (comparable),
           kw_only, __post_init__ for validation.

In [ ]:
@dataclass
class Point2D:
    x: float
    y: float

    def distance_to(self, other: "Point2D") -> float:
        return ((self.x-other.x)**2 + (self.y-other.y)**2) ** 0.5


@dataclass(frozen=True)               # immutable; __hash__ auto-generated
class ImmutablePoint:
    x: float
    y: float


@dataclass(order=True)                # generates __lt__, __le__, __gt__, __ge__
class Version:
    major: int
    minor: int
    patch: int = 0

    def __str__(self) -> str:
        return f"{self.major}.{self.minor}.{self.patch}"


@dataclass
class Order:
    id: int
    items: list[str] = field(default_factory=list)     # safe mutable default
    discount: float  = field(default=0.0, repr=False)  # hide from repr
    _total: float    = field(default=0.0, init=False, repr=False)  # computed

    def __post_init__(self) -> None:
        """Validate and compute after __init__ runs."""
        if not (0 <= self.discount <= 1):
            raise ValueError("discount must be between 0 and 1")

    @property
    def item_count(self) -> int:
        return len(self.items)


@dataclass(slots=True)                # 3.10+ — no __dict__, ~50% less memory
class Coord3D:
    x: float; y: float; z: float = 0.0


def demo_dataclasses() -> None:
    p1 = Point2D(0, 0); p2 = Point2D(3, 4)
    assert abs(p1.distance_to(p2) - 5.0) < 1e-9
    assert p1 != p2   # __eq__ auto-generated

    ip = ImmutablePoint(1.0, 2.0)
    try:
        ip.x = 99      # AttributeError — frozen
    except (AttributeError, TypeError):
        pass
    assert {ip, ImmutablePoint(1.0, 2.0)} == {ip}   # hashable

    v1, v2, v3 = Version(1,2,3), Version(2,0), Version(1,3)
    assert sorted([v1,v2,v3]) == [v1, v3, v2]       # order=True

    o = Order(1, ["apple","banana"], discount=0.1)
    assert o.item_count == 2
    assert "discount" not in repr(o)                # repr=False
    try:
        Order(2, discount=1.5)                      # __post_init__ validates
    except ValueError:
        pass

    # Introspect fields
    f_names = [f.name for f in fields(o)]
    assert "items" in f_names

    print("Part 7 (dataclasses): ✓")

## PART 8 — DESCRIPTORS
Mental model: a descriptor is a CLASS that intercepts attribute access on
  ANOTHER class. It's the machinery behind @property, methods, classmethod,
  staticmethod, and every ORM column field.

Lookup protocol (obj.attr):
  1. Data descriptor in type(obj).__mro__   → __get__ (has __set__ or __delete__)
  2. obj.__dict__["attr"]
  3. Non-data descriptor in type(obj).__mro__  → __get__ (only __get__)
  4. AttributeError

In [ ]:
class Validated:
    """
    DATA descriptor: validates on both read and write.
    One descriptor instance handles ALL instances of the owning class.
    """
    def __set_name__(self, owner: type, name: str) -> None:
        self._attr = "_" + name              # where we store the actual value

    def __get__(self, obj: Any, objtype: type = None) -> Any:
        if obj is None: return self          # accessed on class, return descriptor
        return getattr(obj, self._attr, None)

    def __set__(self, obj: Any, value: Any) -> None:
        self._validate(value)
        setattr(obj, self._attr, value)

    def _validate(self, value: Any) -> None:  # override in subclasses
        pass


class PositiveFloat(Validated):
    def _validate(self, value: Any) -> None:
        if not isinstance(value, (int,float)) or value <= 0:
            raise ValueError(f"{self._attr[1:]} must be a positive number, got {value!r}")


class NonEmptyStr(Validated):
    def _validate(self, value: Any) -> None:
        if not isinstance(value, str) or not value.strip():
            raise ValueError(f"{self._attr[1:]} must be a non-empty string")


class Product:
    """Each descriptor governs ONE field; reused across multiple fields and classes."""
    name     = NonEmptyStr()
    price    = PositiveFloat()
    quantity = PositiveFloat()

    def __init__(self, name: str, price: float, qty: float) -> None:
        self.name, self.price, self.quantity = name, price, qty  # triggers descriptors

    @property
    def total(self) -> float:
        return self.price * self.quantity


class LazyProperty:
    """
    Non-data descriptor: computes value ONCE then stores in instance.__dict__.
    Because it only has __get__ (non-data), instance.__dict__ takes priority
    on subsequent accesses — the expensive computation runs exactly once.
    """
    def __init__(self, fn) -> None:
        self._fn = fn
        self._name = fn.__name__

    def __get__(self, obj: Any, objtype: type = None) -> Any:
        if obj is None: return self
        val = self._fn(obj)
        obj.__dict__[self._name] = val  # store in instance dict
        return val


class Circle2:
    def __init__(self, r: float) -> None: self.r = r

    @LazyProperty
    def area(self) -> float:
        print("computing area once...")
        return 3.14159 * self.r ** 2


def demo_descriptors() -> None:
    p = Product("Widget", 9.99, 10)
    assert abs(p.total - 99.9) < 0.01
    p.price = 5.00
    assert abs(p.total - 50.0) < 0.01

    for bad in (0, -1, "hello"):
        try:   Product("x", bad, 1)
        except ValueError: pass

    for bad in ("", "  "):
        try:   Product(bad, 1.0, 1)
        except ValueError: pass

    # LazyProperty — runs once
    c = Circle2(5)
    _ = c.area    # computes
    _ = c.area    # reads from c.__dict__; no recompute
    assert "area" in c.__dict__

    print("Part 8 (Descriptors): ✓")

## PART 9 — __slots__
Mental model: swap the per-instance __dict__ (a hash table — ~400 bytes) for
  a fixed C-array of value slots — ~56 bytes overhead instead of ~400+.
  Also blocks accidental attribute creation (typo protection).

In [ ]:
class SlottedPoint:
    __slots__ = ("x", "y")
    def __init__(self, x: float, y: float) -> None:
        self.x, self.y = x, y


class DictPoint:
    def __init__(self, x: float, y: float) -> None:
        self.x, self.y = x, y


def demo_slots() -> None:
    import sys
    sp = SlottedPoint(1, 2)
    dp = DictPoint(1, 2)

    assert sp.x == 1 and sp.y == 2
    assert not hasattr(sp, "__dict__")        # no dict

    try:
        sp.z = 3                              # AttributeError — blocked
        raise AssertionError("should have blocked")
    except AttributeError:
        pass

    dp.z = 3                                  # silently creates attribute
    assert dp.z == 3

    # Memory: slotted is significantly smaller
    slotted_size = sys.getsizeof(sp)
    dicted_size  = sys.getsizeof(dp) + sys.getsizeof(dp.__dict__)
    assert slotted_size < dicted_size         # slots win on memory

    print(f"  Slotted: {slotted_size}B  Dicted: {dicted_size}B")
    print("Part 9 (__slots__): ✓")

## PART 10 — __init_subclass__
Mental model: a hook called on the BASE class whenever a SUBCLASS is defined.
  Enables auto-registration, constraint checking, and keyword configuration
  at class-definition time — without metaclass complexity.

In [ ]:
class Serializer:
    """
    Auto-registry: every subclass is registered under a key.
    Subclasses declare their format with a keyword argument.
    """
    _registry: ClassVar[dict[str, type["Serializer"]]] = {}

    def __init_subclass__(cls, fmt: str = "", **kwargs: Any) -> None:
        super().__init_subclass__(**kwargs)   # cooperative — pass kwargs up MRO
        key = fmt or cls.__name__.lower()
        Serializer._registry[key] = cls

    def serialize(self, data: Any) -> str:
        raise NotImplementedError


class JsonSerializer(Serializer, fmt="json"):
    def serialize(self, data: Any) -> str:
        import json; return json.dumps(data)


class CsvSerializer(Serializer, fmt="csv"):
    def serialize(self, data: Any) -> str:
        return ",".join(str(v) for v in data)


class XmlSerializer(Serializer):               # name used as key: "xmlserializer"
    def serialize(self, data: Any) -> str:
        return f"<data>{data}</data>"


def demo_init_subclass() -> None:
    assert "json" in Serializer._registry
    assert "csv"  in Serializer._registry
    s = Serializer._registry["json"]()
    assert s.serialize({"a":1}) == '{"a": 1}'

    print("Part 10 (__init_subclass__): ✓")

## PART 11 — METACLASSES
Mental model: a metaclass is to a class what a class is to an instance.
  type("MyClass", (Base,), {"method": fn}) creates a class dynamically.
  Custom metaclass: override __new__ to modify the class dict BEFORE
  the class object is created — even before __init_subclass__ runs.

In [ ]:
class SingletonMeta(type):
    """
    Metaclass-based singleton: one instance per class.
    __call__ on the metaclass controls what happens when you write ClassName().
    """
    _instances: dict[type, Any] = {}

    def __call__(cls, *args: Any, **kwargs: Any) -> Any:
        if cls not in cls._instances:
            cls._instances[cls] = super().__call__(*args, **kwargs)
        return cls._instances[cls]


class AppConfig(metaclass=SingletonMeta):
    def __init__(self) -> None:
        self.debug = False


class ValidatingMeta(type):
    """
    Enforce that every method in a subclass has a return type annotation.
    Runs at class DEFINITION time — zero runtime overhead.
    """
    def __new__(mcs, name: str, bases: tuple, namespace: dict, **kwargs: Any):
        cls = super().__new__(mcs, name, bases, namespace)
        if bases:                                    # skip the base class
            for attr, val in namespace.items():
                if callable(val) and not attr.startswith("_"):
                    if not hasattr(val, "__annotations__"):
                        pass  # no annotations object at all — OK
                    elif "return" not in (val.__annotations__ or {}):
                        raise TypeError(
                            f"{name}.{attr} must have a return type annotation"
                        )
        return cls


class StrictBase(metaclass=ValidatingMeta):
    pass


def demo_metaclasses() -> None:
    # Singleton
    c1, c2 = AppConfig(), AppConfig()
    assert c1 is c2         # same instance

    # ValidatingMeta
    class GoodModel(StrictBase):
        def process(self) -> str:    # has return annotation ✓
            return "ok"

    try:
        class BadModel(StrictBase):
            def process(self):       # missing return annotation → TypeError
                pass
    except TypeError:
        pass

    print("Part 11 (Metaclasses): ✓")

## PART 12 — CLASSIC DESIGN PATTERNS IN PYTHON

`── FACTORY PATTERN ──────────────────────────────────────────────────────────`

In [ ]:
class Animal(ABC):
    @abstractmethod
    def speak(self) -> str: ...

class AnimalDog(Animal):
    def speak(self) -> str: return "Woof"

class AnimalCat(Animal):
    def speak(self) -> str: return "Meow"

def animal_factory(kind: str) -> Animal:
    """Factory function — creates objects without exposing creation logic."""
    mapping = {"dog": AnimalDog, "cat": AnimalCat}
    cls = mapping.get(kind.lower())
    if cls is None:
        raise ValueError(f"Unknown animal: {kind}")
    return cls()

`── OBSERVER PATTERN ─────────────────────────────────────────────────────────`

In [ ]:
class EventBus:
    """
    Publish-subscribe: producers emit events; consumers subscribe to types.
    Uses weakref so dead consumers are automatically unsubscribed.
    """
    def __init__(self) -> None:
        self._handlers: dict[str, list] = defaultdict(list)

    def subscribe(self, event: str, handler) -> None:
        self._handlers[event].append(weakref.ref(handler))

    def publish(self, event: str, data: Any = None) -> None:
        live = []
        for ref in self._handlers[event]:
            h = ref()
            if h is not None:
                h(data); live.append(ref)
        self._handlers[event] = live   # prune dead refs

`── MIXIN PATTERN ─────────────────────────────────────────────────────────────`

In [ ]:
class JSONMixin:
    """Adds .to_json() to any class. No inheritance from the class needed."""
    def to_json(self) -> str:
        import json
        return json.dumps({
            k: v for k, v in self.__dict__.items() if not k.startswith("_")
        })


class TimestampMixin:
    """Adds created_at to any class."""
    def __init__(self, *args: Any, **kwargs: Any) -> None:
        import time
        super().__init__(*args, **kwargs)      # cooperative
        self.created_at = time.time()


@dataclass
class User(JSONMixin):
    name: str
    email: str


def demo_patterns() -> None:
    # Factory
    d = animal_factory("dog")
    assert d.speak() == "Woof"
    try: animal_factory("fish")
    except ValueError: pass

    # Observer
    bus = EventBus()
    received = []
    def handler(data): received.append(data)  # noqa: E731 (demo)
    bus.subscribe("order.created", handler)
    bus.publish("order.created", {"id": 42})
    assert received == [{"id": 42}]

    # Mixin
    u = User("Ada", "ada@example.com")
    import json
    parsed = json.loads(u.to_json())
    assert parsed["name"] == "Ada"

    print("Part 12 (Design Patterns): ✓")

## MAIN

In [ ]:
def main() -> None:
    print("=" * 70)
    print("OOP COMPLETE — python oop_complete.py")
    print("=" * 70)
    demo_encapsulation()
    demo_inheritance()
    demo_mro()
    demo_abstraction()
    demo_polymorphism()
    demo_dunders()
    demo_class_static()
    demo_dataclasses()
    demo_descriptors()
    demo_slots()
    demo_init_subclass()
    demo_metaclasses()
    demo_patterns()
    print("-" * 70)
    print("All OOP demos passed ✔")

## ═══  EXHAUSTIVE NOTEBOOK  — classmethod · staticmethod · variables  ═══════

In [ ]:
def sep(t): print(f"\n{'═'*64}\n  {t}\n{'═'*64}")

## SECTION A — CLASS VARIABLES vs INSTANCE VARIABLES

Mental model

class variable  = ONE object stored on the class dict — shared by ALL instances.
instance variable = stored in each instance's __dict__ — unique per instance.

Reading  → Python checks instance.__dict__ FIRST, then the class dict.
Writing  → ALWAYS creates / updates an instance variable (never touches the
            class variable unless you explicitly write ClassName.var = ...).

This asymmetry is the source of the most common OOP gotcha.

In [ ]:
def notebook_class_vs_instance_vars() -> None:

## A · Class Variables vs Instance Variables

In [ ]:
# ── A1. Basic: shared class variable ──────────────────────────────────
    # class_count is shared across ALL Counter instances
    class Counter:
        class_count = 0          # class variable — ONE copy for ALL instances

        def __init__(self, name):
            Counter.class_count += 1   # update CLASS variable explicitly
            self.name = name            # instance variable — unique per object

    c1 = Counter("a")
    c2 = Counter("b")
    c3 = Counter("c")
    print(f"Counter.class_count = {Counter.class_count}")   # 3 — shared
    print(f"c1.class_count      = {c1.class_count}")        # 3 — reads from class

    # ── A2. THE GOTCHA: writing through an instance creates an INSTANCE VAR ──
    #
    # GOTCHA: instance writes NEVER touch the class variable.
    #   c1.class_count += 1  is really:
    #     temp = c1.class_count  (reads from class → 3)
    #     c1.class_count = temp + 1  (CREATES new instance var = 4)
    #   The class-level class_count is untouched!

    c1.class_count += 1          # GOTCHA: creates c1.__dict__["class_count"] = 4
    print(f"\nAfter c1.class_count += 1:")
    print(f"  Counter.class_count = {Counter.class_count}")   # still 3 (class untouched)
    print(f"  c1.class_count      = {c1.class_count}")        # 4 (instance shadow)
    print(f"  c2.class_count      = {c2.class_count}")        # 3 (reads class)
    print(f"  'class_count' in c1.__dict__: {'class_count' in c1.__dict__}")   # True (shadowed)
    print(f"  'class_count' in c2.__dict__: {'class_count' in c2.__dict__}")   # False

    # ── A3. THE MUTABLE CLASS VARIABLE TRAP (the worst gotcha) ───────────────
    #
    # GOTCHA: a MUTABLE class variable (list/dict) is shared across ALL
    # instances. Appending via self.attr mutates the SHARED object!

    class BadTeam:
        members = []             # ONE list shared by ALL instances

        def add(self, name):
            self.members.append(name)   # mutates the SHARED class list!

    t1, t2 = BadTeam(), BadTeam()
    t1.add("Alice")
    t2.add("Bob")
    print(f"\nBadTeam GOTCHA:")
    print(f"  t1.members = {t1.members}")   # ['Alice', 'Bob'] — t2's data leaked!
    print(f"  t2.members = {t2.members}")   # ['Alice', 'Bob'] — same list!
    print(f"  t1.members is t2.members: {t1.members is t2.members}")   # True

    # FIX: create the mutable in __init__ (instance variable)
    class GoodTeam:
        def __init__(self):
            self.members = []    # fresh list per instance

        def add(self, name):
            self.members.append(name)

    g1, g2 = GoodTeam(), GoodTeam()
    g1.add("Alice")
    g2.add("Bob")
    print(f"\nGoodTeam FIX:")
    print(f"  g1.members = {g1.members}")   # ['Alice'] ✓
    print(f"  g2.members = {g2.members}")   # ['Bob']   ✓

    # ── A4. Class variables with inheritance ─────────────────────────────────
    #
    # Subclass INHERITS the parent's class variable; writing to the subclass
    # creates a SUBCLASS-LEVEL variable, not an instance variable.

    class Animal:
        kingdom = "Animalia"
        population = 0

        def __init__(self):
            Animal.population += 1   # shared counter: all animals

    class Dog(Animal):
        species = "Canis lupus familiaris"

    class Cat(Animal):
        species = "Felis catus"

    d, c = Dog(), Cat()
    print(f"\nInheritance class vars:")
    print(f"  Animal.kingdom = {Animal.kingdom!r}")   # 'Animalia'
    print(f"  Dog.kingdom    = {Dog.kingdom!r}")      # 'Animalia' — inherited
    print(f"  Animal.population = {Animal.population}")   # 2 (Dog + Cat)

    Dog.kingdom = "Overridden"   # creates Dog-level class var
    print(f"  After Dog.kingdom = 'Overridden':")
    print(f"    Dog.kingdom    = {Dog.kingdom!r}")    # 'Overridden'
    print(f"    Animal.kingdom = {Animal.kingdom!r}") # 'Animalia' — untouched
    print(f"    Cat.kingdom    = {Cat.kingdom!r}")    # 'Animalia' — unaffected

    # ── A5. Inspect with __dict__ and vars() ─────────────────────────────────
    print(f"\nInspection tools:")
    class Inspectable:
        cv = "class-level"
        def __init__(self): self.iv = "instance-level"

    obj = Inspectable()
    print(f"  obj.__dict__   = {obj.__dict__}")              # {{'iv': 'instance-level'}}
    print(f"  Inspectable.__dict__ has 'cv': {'cv' in Inspectable.__dict__}")  # True
    print(f"  obj.__dict__ has 'cv':         {'cv' in obj.__dict__}")          # False
    print(f"  vars(obj) = {vars(obj)}")                      # same as obj.__dict__

## SECTION B — @classmethod

Mental model

@classmethod receives the CLASS (cls) as its first argument instead of
an instance.  This makes it POLYMORPHIC: when called on a subclass,
cls IS the subclass — so you can create the right type.

Three canonical uses:
  1. Alternative constructors (from_json, from_csv, from_env …)
  2. Accessing/modifying class-level state without a hardcoded class name
  3. Factory methods that respect subclassing (cls() not ClassName())

Can be called on the CLASS:     MyClass.create(...)
Can be called on an INSTANCE:   obj.create(...)  ← cls is still the class

In [ ]:
def notebook_classmethod() -> None:

## B · @classmethod — Polymorphic Factories & Class State

In [ ]:
# ── B1. Basic: alternative constructor ────────────────────────────────
    class Color:
        def __init__(self, r, g, b):
            self.r, self.g, self.b = r, g, b

        @classmethod
        def from_hex(cls, hex_str):
            """Alternative constructor — cls() not Color() so subclasses work."""
            h = hex_str.lstrip("#")
            return cls(int(h[0:2],16), int(h[2:4],16), int(h[4:6],16))

        @classmethod
        def from_name(cls, name):
            names = {"red":(255,0,0), "green":(0,255,0), "blue":(0,0,255)}
            r,g,b = names[name.lower()]
            return cls(r,g,b)

        def __repr__(self):
            return f"{type(self).__name__}({self.r},{self.g},{self.b})"

    c1 = Color.from_hex("#FF8040")
    c2 = Color.from_name("blue")
    print(f"from_hex: {c1}")
    print(f"from_name: {c2}")

    # ── B2. POLYMORPHISM — cls is the ACTUAL class called on ─────────────
    class PremiumColor(Color):
        def __init__(self, r, g, b):
            super().__init__(r, g, b)
            self.premium = True

    pc = PremiumColor.from_hex("#FF0000")   # cls = PremiumColor !
    print(f"\nPolymorphic classmethod:")
    print(f"  type(pc)       = {type(pc).__name__}")   # PremiumColor, NOT Color
    print(f"  isinstance(pc, PremiumColor): {isinstance(pc, PremiumColor)}")   # True
    print(f"  pc.premium:    {pc.premium}")             # True ✓

    # GOTCHA: if you hard-coded Color() instead of cls(), subclass gets wrong type
    class BrokenColor(Color):
        @classmethod
        def from_hex(cls, hex_str):
            h = hex_str.lstrip("#")
            return Color(int(h[0:2],16), int(h[2:4],16), int(h[4:6],16))   # WRONG: Color not cls

    class SubBroken(BrokenColor): pass
    b = SubBroken.from_hex("#FFFFFF")
    print(f"\nHardcoded class name GOTCHA:")
    print(f"  type(b) = {type(b).__name__}")   # Color — not SubBroken!

    # ── B3. @classmethod for class-level state ────────────────────────────
    class Registry:
        _instances = {}     # class-level registry

        def __init__(self, name):
            self.name = name
            Registry._instances[name] = self   # register on creation

        @classmethod
        def get(cls, name):
            """Retrieve registered instance — uses class-level dict."""
            return cls._instances.get(name)

        @classmethod
        def count(cls):
            return len(cls._instances)

        @classmethod
        def clear(cls):
            cls._instances.clear()

    r1 = Registry("svc-a")
    r2 = Registry("svc-b")
    print(f"\nClass-level state:")
    print(f"  Registry.count() = {Registry.count()}")        # 2
    print(f"  Registry.get('svc-a') = {Registry.get('svc-a').name!r}")
    Registry.clear()
    print(f"  After clear: {Registry.count()}")              # 0

    # ── B4. Calling classmethod on an INSTANCE ─────────────────────────────
    #
    # GOTCHA: calling a classmethod on an instance passes the CLASS, not the instance.
    obj = Color(255, 0, 0)
    c_from_instance = obj.from_name("green")   # cls = Color, not any subclass
    print(f"\nCalling classmethod on instance:")
    print(f"  obj.from_name('green') → {c_from_instance}  (cls={type(c_from_instance).__name__})")

    # ── B5. classmethod in __init_subclass__ ─────────────────────────────
    class Plugin:
        _plugins = {}

        def __init_subclass__(cls, key="", **kwargs):
            super().__init_subclass__(**kwargs)
            Plugin._plugins[key or cls.__name__.lower()] = cls

        @classmethod
        def create(cls, key):
            return cls._plugins[key]()

    class JsonPlugin(Plugin, key="json"): pass
    class CsvPlugin(Plugin,  key="csv"):  pass

    print(f"\nPlugin factory via classmethod:")
    print(f"  Plugin.create('json') → {type(Plugin.create('json')).__name__}")

## SECTION C — @staticmethod

Mental model

@staticmethod is a plain function namespaced inside a class.
It receives NO implicit first argument (no self, no cls).
It is NOT polymorphic — it does not know which class called it.

Use when the function:
  • logically belongs to the class (conceptually grouped)
  • does NOT need access to the instance OR the class
  • is a pure utility / helper

Comparison:
  instance method → needs self (instance state)
  classmethod     → needs cls  (class state, polymorphism)
  staticmethod    → needs nothing from the class system

In [ ]:
def notebook_staticmethod() -> None:

## C · @staticmethod — Namespaced Utilities

In [ ]:
# ── C1. Basic: pure utility ────────────────────────────────────────────
    class MathHelper:
        @staticmethod
        def clamp(value, lo, hi):
            """Pure utility — no self or cls needed."""
            return max(lo, min(hi, value))

        @staticmethod
        def lerp(a, b, t):
            return a + (b - a) * t

    print(f"clamp(15, 0, 10) = {MathHelper.clamp(15, 0, 10)}")   # 10
    print(f"lerp(0, 100, 0.25) = {MathHelper.lerp(0, 100, 0.25)}")  # 25.0

    # ── C2. staticmethod is NOT polymorphic ────────────────────────────────
    #
    # GOTCHA: staticmethod ignores WHICH class/subclass it's called on.
    # The function body can't access cls — it can't create the "right" subtype.

    class Shape:
        @staticmethod
        def validate_sides(n):
            return n >= 3

        @classmethod
        def create_regular(cls, sides):
            """classmethod CAN create the right subclass; staticmethod cannot."""
            if not cls.validate_sides(sides):
                raise ValueError(f"need ≥ 3 sides, got {sides}")
            return cls()   # cls = whatever subclass called this

    class Triangle(Shape): pass
    class Pentagon(Shape): pass

    t = Triangle.create_regular(3)   # cls = Triangle → Triangle()
    p = Pentagon.create_regular(5)   # cls = Pentagon → Pentagon()
    print(f"\nPolymorphic classmethod works: {type(t).__name__}, {type(p).__name__}")

    # validate_sides on subclass: same function, no polymorphism
    print(f"Triangle.validate_sides(3) = {Triangle.validate_sides(3)}")
    print(f"Pentagon.validate_sides(3) = {Pentagon.validate_sides(3)}")
    # Both call the SAME function — no cls differentiation

    # ── C3. staticmethod inheritance — it IS inherited, just not polymorphic ─
    class Parent:
        @staticmethod
        def utility():
            return "parent utility"

    class Child(Parent):
        pass   # inherits utility()

    print(f"\nInheritance:")
    print(f"  Parent.utility() = {Parent.utility()!r}")
    print(f"  Child.utility()  = {Child.utility()!r}")   # same function
    print(f"  Parent.utility is Child.utility: {Parent.utility is Child.utility}")  # True

    # ── C4. staticmethod vs module-level function — when to choose ─────────
    #
    # Use staticmethod when:
    #   ✓ the function is conceptually "part of the class API"
    #   ✓ you want it callable as ClassName.helper() without instantiating
    #   ✓ subclasses may want to override it (unlike a module function)
    #
    # Use a plain module-level function when:
    #   ✓ the function is truly general-purpose
    #   ✓ no conceptual link to a specific class

    class Validator:
        @staticmethod
        def is_valid_email(email):
            return "@" in email and "." in email.split("@")[-1]

        @staticmethod
        def is_valid_age(age):
            return isinstance(age, int) and 0 <= age <= 150

    print(f"\nValidator utilities (no instantiation needed):")
    print(f"  is_valid_email('a@b.com') = {Validator.is_valid_email('a@b.com')}")
    print(f"  is_valid_age(25)          = {Validator.is_valid_age(25)}")
    print(f"  is_valid_age(-1)          = {Validator.is_valid_age(-1)}")

## SECTION D — SIDE-BY-SIDE COMPARISON & INHERITANCE TABLE

Behaviour across: definition · calling on class · calling on instance ·
first arg · polymorphism · override in subclass

In [ ]:
def notebook_method_comparison() -> None:

## D · Method Type Comparison — Side-by-Side

In [ ]:
class Base:
        tag = "base"

        def instance_method(self):
            return f"instance: self.tag={self.tag}"

        @classmethod
        def class_method(cls):
            return f"classmethod: cls.tag={cls.tag}"

        @staticmethod
        def static_method():
            return "staticmethod: no self/cls"

    class Sub(Base):
        tag = "sub"   # class variable override

        def instance_method(self):
            parent = super().instance_method()
            return f"Sub.instance ({parent})"

        @classmethod
        def class_method(cls):
            return f"Sub.classmethod: cls={cls.__name__}, tag={cls.tag}"

        # staticmethod NOT overridden — inherited as-is

    b, s = Base(), Sub()

    print("─── Calling on CLASS ────────────────────────────────────────────")
    print(f"  Base.class_method()   → {Base.class_method()!r}")
    print(f"  Sub.class_method()    → {Sub.class_method()!r}")
    print(f"  Base.static_method()  → {Base.static_method()!r}")
    print(f"  Sub.static_method()   → {Sub.static_method()!r}")

    print("\n─── Calling on INSTANCE ─────────────────────────────────────────")
    print(f"  b.instance_method()  → {b.instance_method()!r}")
    print(f"  s.instance_method()  → {s.instance_method()!r}")
    print(f"  b.class_method()     → {b.class_method()!r}")   # cls=Base
    print(f"  s.class_method()     → {s.class_method()!r}")   # cls=Sub ← polymorphic!
    print(f"  b.static_method()    → {b.static_method()!r}")
    print(f"  s.static_method()    → {s.static_method()!r}")  # same function as Base

    print("\n─── First argument received ─────────────────────────────────────")
    class Inspector:
        @classmethod
        def show_cls(cls):    print(f"  classmethod received: cls={cls}")
        @staticmethod
        def show_none():      print(f"  staticmethod received: (nothing)")
        def show_self(self):  print(f"  instance method received: self={self}")

    i = Inspector()
    Inspector.show_cls()     # cls = <class 'Inspector'>
    i.show_cls()             # cls = <class 'Inspector'> (same even on instance)
    Inspector.show_none()    # nothing
    i.show_none()            # nothing (same even on instance)
    i.show_self()            # self = <Inspector object>

    print("\n─── Polymorphism check ──────────────────────────────────────────")
    class Factory:
        @classmethod
        def make(cls): return cls()           # ← cls = actual called class

        @staticmethod
        def make_static(): return Factory()   # ← hardcoded, never polymorphic

    class Widget(Factory): pass

    w_cls = Widget.make()        # cls = Widget → Widget()
    w_sta = Widget.make_static() # always Factory()
    print(f"  Widget.make()        → {type(w_cls).__name__}")   # Widget ✓
    print(f"  Widget.make_static() → {type(w_sta).__name__}")   # Factory ✗

    print("\n─── Summary Table ───────────────────────────────────────────────")
    rows = [
        ("", "instance method", "classmethod", "staticmethod"),
        ("decorator",    "none",         "@classmethod",  "@staticmethod"),
        ("first arg",    "self",          "cls",           "none"),
        ("class access", "self.__class__","cls (direct)",  "ClassName.X only"),
        ("polymorphic?", "yes (via self)","yes (cls=sub)", "no"),
        ("call on class","TypeError",     "✓ works",       "✓ works"),
        ("use case",     "instance ops",  "factories/alt", "pure utilities"),
    ]
    for r in rows:
        print(f"  {r[0]:16s} | {r[1]:20s} | {r[2]:20s} | {r[3]}")

## SECTION E — @property INHERITANCE & GOTCHAS

Mental model

@property is a DATA DESCRIPTOR. In subclasses:
  • To OVERRIDE just the getter: @property must be redefined completely.
  • To OVERRIDE just the setter: use @Parent.attr.setter — but only if
    the getter is NOT also overridden in the subclass.
  • To override BOTH: redefine @property + @name.setter from scratch.

> ⚠️ **GOTCHA: if you define a new @property (getter only) in a subclass,**
  the parent's setter is NOT inherited for that property.
  Attempting to set raises AttributeError.

In [ ]:
def notebook_property_inheritance() -> None:

## E · @property Inheritance Gotchas

In [ ]:
class Base:
        def __init__(self): self._x = 0

        @property
        def x(self): return self._x

        @x.setter
        def x(self, v):
            if v < 0: raise ValueError("x must be non-negative")
            self._x = v

    # ── E1. Override getter AND setter together (correct approach) ─────────
    class DoubledSub(Base):
        @property
        def x(self): return self._x * 2   # new getter

        @x.setter   # ← refers to THIS class's property, not Base's
        def x(self, v):
            if v < 0: raise ValueError
            self._x = v

    ds = DoubledSub(); ds.x = 5
    print(f"DoubledSub.x = {ds.x} (getter doubled: 5*2)")   # 10 ✓

    # ── E2. GOTCHA: override getter only — setter is lost ─────────────────
    class BrokenSub(Base):
        @property
        def x(self):
            return self._x * 3   # new getter, but FORGET the setter

    bs = BrokenSub()
    try:
        bs.x = 5           # AttributeError: can't set attribute
    except AttributeError as e:
        print(f"\nGOTCHA — getter-only override loses setter: {e}")

    # ── E3. Correct: use parent's property as base ─────────────────────────
    class FixedSub(Base):
        @Base.x.getter         # extend only the getter, keep Base's setter
        def x(self):
            return self._x + 100

    fs = FixedSub(); fs.x = 5    # setter from Base works
    print(f"FixedSub.x = {fs.x} (5 + 100 = 105)")   # 105 ✓
    try:
        fs.x = -1              # Base's setter validation still active
    except ValueError:
        print(f"Base setter validation still works in FixedSub ✓")

    # ── E4. ClassVar annotation (type-checker signal) ─────────────────────
    from typing import ClassVar

    class Config:
        DEBUG: ClassVar[bool] = False   # signals to mypy: this is a class var
        def __init__(self, host): self.host = host   # this is an instance var

    c = Config("localhost")
    print(f"\nClassVar annotation:")
    print(f"  Config.DEBUG = {Config.DEBUG}  (class-level)")
    print(f"  c.host       = {c.host!r}  (instance-level)")


def run_classmethod_staticmethod_notebook() -> None:
    notebook_class_vs_instance_vars()
    notebook_classmethod()
    notebook_staticmethod()
    notebook_method_comparison()
    notebook_property_inheritance()
    print("\n" + "═"*64)
    print("  classmethod / staticmethod / variables NOTEBOOK COMPLETE")
    print("═"*64)


if __name__ == "__main__":
    if hasattr(sys.stdout, "reconfigure"):
        sys.stdout.reconfigure(encoding="utf-8")
    main()
    run_classmethod_staticmethod_notebook()

---
## Real-World OOP Scenarios — BuildFast CI/CD Platform


# Python Oop Scenarios

*Run each cell with **Shift+Enter***

Python OOP — Real-World Scenarios
===================================
Scenario system: BuildFast — a SaaS CI/CD platform

Each section: Production problem → Without → With → Where seen → Gotchas

Run: python python_oop_scenarios.py

In [ ]:
from __future__ import annotations
import sys, time, weakref
from abc import ABC, abstractmethod
from dataclasses import dataclass, field
from functools import total_ordering
from typing import Protocol, ClassVar

def sep(t): print(f"\n{'═'*64}\n  {t}\n{'═'*64}")
def h(t):   print(f"\n  ── {t} ──")

## SCENARIO 1 — ENCAPSULATION: USER CREDENTIAL STORAGE

REAL INCIDENT

BuildFast stored API tokens as plain strings on User objects.
A logging library's repr() accidentally dumped the full User object
into Elasticsearch. All 50k user tokens were in the search index.
The fix: encapsulate the secret behind a @property that never reveals
the raw value; the repr only shows a masked version.

In [ ]:
def scenario_encapsulation() -> None:

## SCENARIO 1 · Encapsulation — API Token Leak Incident

In [ ]:
h("WITHOUT — token exposed in repr/str, accessible directly")
    class BadUser:
        def __init__(self, email: str, api_token: str) -> None:
            self.email     = email
            self.api_token = api_token   # public! any repr/log sees it

    user = BadUser("alice@buildfast.io", "bfk_supersecrettoken123456")
    print(f"Bad user repr: {vars(user)}")  # dumps the full token into logs!
    # An exception traceback or logging library str(user) leaks the token

    h("WITH — encapsulated credential, safe repr")
    class User:
        def __init__(self, email: str, api_token: str) -> None:
            self.email      = email
            self.__api_token = api_token   # name-mangled: _User__api_token

        @property
        def api_token_masked(self) -> str:
            """Safe view: only the last 4 chars for verification."""
            token = self.__api_token
            return "bfk_" + "*" * (len(token) - 8) + token[-4:]

        def verify_token(self, provided: str) -> bool:
            """Constant-time comparison — no early exit leaks timing info."""
            import hmac
            return hmac.compare_digest(self.__api_token, provided)

        def __repr__(self) -> str:
            return f"User(email={self.email!r}, token={self.api_token_masked!r})"

    u = User("alice@buildfast.io", "bfk_supersecrettoken123456")
    print(f"Safe repr:  {u}")            # shows masked token
    print(f"Verify OK:  {u.verify_token('bfk_supersecrettoken123456')}")
    print(f"Verify bad: {u.verify_token('bfk_wrongtoken')}")
    try:
        print(u.__api_token)            # AttributeError — name-mangled
    except AttributeError as e:
        print(f"Direct access blocked: {e}")
    print(f"But accessible via mangled name (internal use): "
          f"{u._User__api_token[:8]}...")   # for internal class methods

    h("@property for validated attributes — pipeline priority")
    class Pipeline:
        VALID_PRIORITIES = {"low", "normal", "high", "critical"}

        def __init__(self, name: str, priority: str = "normal") -> None:
            self.name = name
            self.priority = priority     # calls the setter below

        @property
        def priority(self) -> str: return self._priority

        @priority.setter
        def priority(self, value: str) -> None:
            if value not in self.VALID_PRIORITIES:
                raise ValueError(f"Priority must be one of {self.VALID_PRIORITIES}")
            self._priority = value

    p = Pipeline("build-frontend", "high")
    print(f"\nPipeline priority: {p.priority}")
    try:
        p.priority = "SUPER_URGENT"   # rejected at assignment time
    except ValueError as e:
        print(f"Validation on assignment: {e}")

    h("WHERE IN FRAMEWORKS")
    print("""
  Django User model: password field stores hash, never the plaintext
    user.password = "raw"        ← DON'T DO THIS in Django
    user.set_password("raw")     ← hashes via PBKDF2, safe

  Pydantic SecretStr: wraps sensitive data, masks in repr
    class Settings(BaseSettings):
        database_url: SecretStr   ← str(settings.database_url) = '**hidden**'

  SQLAlchemy: hybrid_property for computed columns
    @hybrid_property
    def full_name(self): return f"{self.first} {self.last}"

  FastAPI: Depends() with OAuth2 injects a verified user object,
    not the raw token — callers never see the token bytes
""")

## SCENARIO 2 — INHERITANCE & MRO: PIPELINE RUNNER HIERARCHY

REAL DESIGN DECISION at BuildFast

BuildFast runs pipelines on multiple environments: Docker (default),
Kubernetes (enterprise), LocalExec (dev mode), and WASM (sandboxed).
All share common lifecycle: validate → setup → run → teardown → report.
But each environment has wildly different setup/teardown code.

Without a proper hierarchy, the run() function had a 150-line
if/elif on runner_type. Adding a new runner required editing the
core run() logic — the riskiest operation each sprint.

In [ ]:
def scenario_inheritance_mro() -> None:

## SCENARIO 2 · Inheritance & MRO — Pipeline Runner Hierarchy

In [ ]:
h("WITHOUT — 150-line if/elif on runner_type")
    def run_pipeline_BAD(pipeline: dict, runner_type: str) -> dict:
        if runner_type == "docker":
            print(f"    [docker] pull image, create container")
            result = f"docker_result_{pipeline['name']}"
            print(f"    [docker] remove container")
        elif runner_type == "k8s":
            print(f"    [k8s] create pod spec, apply to cluster")
            result = f"k8s_result_{pipeline['name']}"
            print(f"    [k8s] delete pod")
        elif runner_type == "local":
            print(f"    [local] set env vars")
            result = f"local_result_{pipeline['name']}"
            print(f"    [local] clean temp files")
        else:
            raise ValueError(f"Unknown runner: {runner_type}")
        return {"status": "ok", "result": result}
    # ✗ Adding WASM runner = edit this function
    # ✗ Docker logic and K8s logic are in the same function

    h("WITH — Template Method + inheritance hierarchy")
    class PipelineRunner(ABC):
        """Template method: fixed lifecycle, overridable steps."""

        def run(self, pipeline: dict) -> dict:
            """Invariant skeleton: validate → setup → execute → teardown → report."""
            self._validate(pipeline)
            try:
                self._setup(pipeline)
                result = self._execute(pipeline)
                return self._report(pipeline, result, error=None)
            except Exception as e:
                return self._report(pipeline, result=None, error=e)
            finally:
                self._teardown(pipeline)   # ALWAYS runs

        def _validate(self, p: dict) -> None:
            if not p.get("steps"):
                raise ValueError("Pipeline must have at least one step")

        @abstractmethod
        def _setup(self, p: dict) -> None: ...

        @abstractmethod
        def _execute(self, p: dict) -> str: ...

        @abstractmethod
        def _teardown(self, p: dict) -> None: ...

        def _report(self, p: dict, result, error) -> dict:
            return {
                "pipeline": p["name"],
                "runner":   type(self).__name__,
                "status":   "error" if error else "success",
                "result":   str(result) if result else str(error),
            }

    class DockerRunner(PipelineRunner):
        def _setup(self, p):    print(f"    [Docker] pull {p.get('image','ubuntu')}, create container")
        def _execute(self, p):  return f"container_{p['name']}_output"
        def _teardown(self, p): print(f"    [Docker] remove container, cleanup volumes")

    class KubernetesRunner(PipelineRunner):
        def _setup(self, p):    print(f"    [K8s] create pod spec, submit to cluster")
        def _execute(self, p):  return f"pod_{p['name']}_exit_0"
        def _teardown(self, p): print(f"    [K8s] delete pod, free PVC")

    # Adding WASM runner: NEW CLASS ONLY — zero edits to PipelineRunner
    class WasmRunner(PipelineRunner):
        def _setup(self, p):    print(f"    [WASM] instantiate sandbox")
        def _execute(self, p):  return f"wasm_{p['name']}_sandboxed"
        def _teardown(self, p): print(f"    [WASM] destroy sandbox, verify isolation")

    def run_any(pipeline: dict, runner: PipelineRunner) -> dict:
        return runner.run(pipeline)   # no knowledge of which runner

    pipe = {"name": "frontend-ci", "steps": ["lint", "test", "build"]}
    for Runner in [DockerRunner, KubernetesRunner, WasmRunner]:
        r = run_any(pipe, Runner())
        print(f"  {r['runner']}: {r['status']} — {r['result']}")

    h("Mixin for monitoring — cooperative super() with MRO")
    class MonitoringMixin:
        """Adds timing and metrics to any PipelineRunner."""
        def run(self, pipeline: dict) -> dict:
            t0 = time.perf_counter()
            result = super().run(pipeline)   # ← follows MRO, not just DockerRunner
            result["duration_ms"] = round((time.perf_counter() - t0) * 1000, 2)
            print(f"    [Metrics] duration={result['duration_ms']}ms")
            return result

    class MonitoredDockerRunner(MonitoringMixin, DockerRunner):
        """MRO: MonitoredDockerRunner → MonitoringMixin → DockerRunner → PipelineRunner"""
        pass

    r = MonitoredDockerRunner().run(pipe)
    print(f"  Monitored result has duration: {'duration_ms' in r}")

    h("WHERE TEMPLATE METHOD + INHERITANCE IS USED")
    print("""
  Django class-based views: ListView, CreateView, UpdateView all inherit
    View with a dispatch() template method. Override get()/post() only.

  Python unittest: TestCase.run() is a template method. You override
    setUp(), test_*(), tearDown() — the framework calls them in order.

  SQLAlchemy: TypeDecorator lets you subclass to add custom type logic
    while TypeDecorator.process_bind_param/process_result_value are the hooks.

  Celery Task: base Task class has a run() template; override run() for logic,
    on_failure()/on_success() as hooks.
""")

## SCENARIO 3 — PROTOCOL vs ABC: FastAPI DEPENDENCY INJECTION

REAL DESIGN DECISION at BuildFast

BuildFast has a NotificationService that sends emails.
In production: SendGrid. In staging: Mailhog (local SMTP).
In unit tests: in-memory list (no network, no credentials).
In CI: suppress (discard all notifications).

The team debated: ABC with EmailSenderBase subclasses, or Protocol?
They chose Protocol — no import coupling between the service and the
concrete senders. Tests pass a plain fake with no base class.

In [ ]:
def scenario_protocol_vs_abc() -> None:

## SCENARIO 3 · Protocol vs ABC — FastAPI DI for Notifications

In [ ]:
h("WITHOUT Protocol — ABC forces coupling and stubs")
    class EmailSenderABC(ABC):
        @abstractmethod
        def send(self, to: str, subject: str, body: str) -> bool: ...
        @abstractmethod
        def get_stats(self) -> dict: ...     # SendGrid-specific — forces others to stub!
        @abstractmethod
        def validate_domain(self, domain: str) -> bool: ...  # not needed by Mailhog!

    class FakeEmailSenderBad(EmailSenderABC):
        def send(self, to, subject, body): return True
        def get_stats(self):       return {}          # FORCED STUB — never used in tests
        def validate_domain(self, d): return True     # FORCED STUB — never called

    h("WITH Protocol — structural typing, fake needs NO base class")
    class EmailSender(Protocol):
        """Minimal interface: only what NotificationService actually needs."""
        def send(self, to: str, subject: str, body: str) -> bool: ...

    class SendGridSender:
        def send(self, to, subject, body) -> bool:
            print(f"    [SendGrid] → {to}: {subject}")
            return True
        def get_stats(self): return {"sent": 42}       # extra methods are fine
        def validate_domain(self, d): return True       # not in Protocol — invisible

    class MailhogSender:
        def send(self, to, subject, body) -> bool:
            print(f"    [Mailhog] → {to}: {subject}")
            return True
        # No get_stats, no validate_domain — doesn't need them

    class FakeEmailSender:
        """Test double — no ABC, no imports, no stubs."""
        def __init__(self): self.sent: list[dict] = []
        def send(self, to, subject, body) -> bool:
            self.sent.append({"to": to, "subject": subject})
            return True

    class SuppressSender:
        def send(self, to, subject, body) -> bool: return True   # silently discard

    class NotificationService:
        def __init__(self, sender: EmailSender) -> None:
            self._sender = sender

        def notify_build_complete(self, user_email: str, pipeline: str, status: str) -> None:
            self._sender.send(
                to=user_email,
                subject=f"Build {pipeline} {status}",
                body=f"Your pipeline {pipeline!r} completed with status: {status}"
            )

    # Production
    svc_prod = NotificationService(SendGridSender())
    svc_prod.notify_build_complete("alice@co.com", "frontend-ci", "success")

    # Test — no network, no credentials, full control
    fake = FakeEmailSender()
    svc_test = NotificationService(fake)
    svc_test.notify_build_complete("alice@co.com", "frontend-ci", "success")
    assert fake.sent[0]["to"] == "alice@co.com"
    assert "frontend-ci" in fake.sent[0]["subject"]
    print(f"  Test captured: {fake.sent[0]}")
    print(f"  → Zero imports from SendGrid SDK in tests ✓")

    # CI — suppress
    svc_ci = NotificationService(SuppressSender())
    svc_ci.notify_build_complete("*", "*", "skipped")
    print(f"  CI: notifications silently suppressed ✓")

    h("WHERE Protocol vs ABC is used in real systems")
    print("""
  FastAPI Depends():
    async def get_db() → AsyncSession:   ← Protocol: async context manager
    async def get_user(token: OAuth2Token) → User:  ← Protocol for auth

  The key rule (Architect answer):
    USE Protocol  when: you consume the interface, test it with fakes, need
                        zero coupling to third-party libs in tests.
                        ("Protocol for what I consume")

    USE ABC       when: you OWN the hierarchy, share code in the base,
                        want instantiation-time enforcement.
                        ("ABC for what I own and extend")
""")

## SCENARIO 4 — DESCRIPTORS: ORM-STYLE FIELD VALIDATION

REAL DESIGN at BuildFast (mirrors Django/SQLAlchemy field system)

BuildFast defines pipeline configuration models. Every field needs
type checking + range validation. Without descriptors: copy-paste
the validation in __init__ for every field, every class.
With descriptors: write validation ONCE, reuse across all fields.
This is exactly how Django models work internally.

In [ ]:
def scenario_descriptors() -> None:

## SCENARIO 4 · Descriptors — Django-style Field Validation System

In [ ]:
h("WITHOUT — validation copy-pasted in every __init__")
    class BadPipelineConfig:
        def __init__(self, max_parallel: int, timeout_minutes: int, retry_count: int):
            if not isinstance(max_parallel, int) or max_parallel < 1:
                raise ValueError("max_parallel must be int >= 1")
            if not isinstance(timeout_minutes, int) or not (1 <= timeout_minutes <= 1440):
                raise ValueError("timeout_minutes must be 1..1440")
            if not isinstance(retry_count, int) or not (0 <= retry_count <= 10):
                raise ValueError("retry_count must be 0..10")
            self.max_parallel   = max_parallel
            self.timeout_minutes = timeout_minutes
            self.retry_count    = retry_count
    # ✗ Validation is in __init__ ONLY — setting p.max_parallel = -1 after init bypasses it!
    # ✗ Adding a new config class = copy-paste all this validation

    h("WITH — reusable data descriptors (like Django's IntegerField)")
    class IntField:
        """Data descriptor: validates on EVERY assignment, not just __init__."""
        def __init__(self, min_val: int | None = None, max_val: int | None = None):
            self.min = min_val
            self.max = max_val

        def __set_name__(self, owner, name):
            self._attr = f"_{name}"   # storage slot: _max_parallel etc.

        def __get__(self, obj, objtype=None):
            if obj is None: return self          # class-level access → return descriptor
            return getattr(obj, self._attr, None)

        def __set__(self, obj, value):
            if not isinstance(value, int):
                raise TypeError(f"{self._attr[1:]} must be int, got {type(value).__name__}")
            if self.min is not None and value < self.min:
                raise ValueError(f"{self._attr[1:]} must be >= {self.min}, got {value}")
            if self.max is not None and value > self.max:
                raise ValueError(f"{self._attr[1:]} must be <= {self.max}, got {value}")
            setattr(obj, self._attr, value)

    class StringField:
        def __init__(self, max_length: int = 255, choices: list | None = None):
            self.max_length = max_length
            self.choices    = choices

        def __set_name__(self, owner, name): self._attr = f"_{name}"

        def __get__(self, obj, t=None):
            return self if obj is None else getattr(obj, self._attr, "")

        def __set__(self, obj, value):
            if not isinstance(value, str):
                raise TypeError(f"{self._attr[1:]} must be str")
            if len(value) > self.max_length:
                raise ValueError(f"{self._attr[1:]} too long: {len(value)} > {self.max_length}")
            if self.choices and value not in self.choices:
                raise ValueError(f"{self._attr[1:]} must be one of {self.choices}")
            setattr(obj, self._attr, value)

    class PipelineConfig:
        max_parallel    = IntField(min_val=1, max_val=50)
        timeout_minutes = IntField(min_val=1, max_val=1440)
        retry_count     = IntField(min_val=0, max_val=10)
        trigger         = StringField(choices=["push", "pr", "schedule", "manual"])
        name            = StringField(max_length=100)

        def __init__(self, name, max_parallel, timeout_minutes, retry_count, trigger):
            self.name            = name             # calls StringField.__set__
            self.max_parallel    = max_parallel     # calls IntField.__set__
            self.timeout_minutes = timeout_minutes
            self.retry_count     = retry_count
            self.trigger         = trigger

    cfg = PipelineConfig("frontend-ci", max_parallel=4, timeout_minutes=30,
                          retry_count=2, trigger="push")
    print(f"Valid config: {cfg.name}, parallel={cfg.max_parallel}, trigger={cfg.trigger}")

    # Validation fires on EVERY assignment — __init__ AND after
    try:
        cfg.max_parallel = -1    # caught even after __init__!
    except ValueError as e:
        print(f"Post-init validation: {e}")

    try:
        cfg.trigger = "webhook"  # not in choices
    except ValueError as e:
        print(f"Choice validation: {e}")

    h("WHERE DESCRIPTORS ARE USED IN REAL FRAMEWORKS")
    print("""
  Django models:  every model field (IntegerField, CharField, ForeignKey)
    IS a descriptor. `Model.name = "x"` triggers CharField's __set__,
    which marks the instance as dirty for the ORM to track.

  SQLAlchemy ORM: Column() and relationship() are descriptors.
    `user.email = "x"` triggers InstrumentedAttribute.__set__,
    which adds the change to the session's unit-of-work.

  Pydantic v2 (FieldInfo): field validators are descriptor-backed
    under the hood via Rust extension — same concept, compiled.

  Python @property: IS a data descriptor (has __get__ + __set__).
    The difference between @property and a descriptor:
      @property: one field per class definition
      Descriptor class: ONE reusable object validates N fields across N classes
""")

## SCENARIO 5 — DATACLASSES IN APIs: TYPED, VALIDATED PIPELINE RESULTS

REAL USE CASE at BuildFast

BuildFast returns pipeline run results as dicts, then as TypedDicts,
then as dataclasses. Each step solved a real problem. The final form
(frozen dataclass) is immutable: result objects can be safely cached,
passed across threads, and used as dict keys for deduplication.

In [ ]:
def scenario_dataclasses() -> None:

## SCENARIO 5 · Dataclasses — Typed Pipeline Results at BuildFast

In [ ]:
h("Evolution 1: plain dict — works but easy to mistype keys")
    run_result_dict = {
        "pipeline_id": "pipe_001",
        "status": "success",
        "duration_ms": 12340,
        "steps_run": ["lint", "test", "build"],
    }
    # ✗ run_result_dict["statsu"]  ← typo surfaces at runtime, not import time

    h("Evolution 2: dataclass — IDE autocomplete, type checking, __repr__")
    from dataclasses import dataclass

    @dataclass
    class PipelineResult:
        pipeline_id:  str
        status:       str
        duration_ms:  int
        steps_run:    list[str]
        error_message: str | None = None

    r = PipelineResult("pipe_001", "success", 12340, ["lint", "test", "build"])
    print(f"Dataclass repr: {r}")
    print(f"Equality works: {r == PipelineResult('pipe_001','success',12340,['lint','test','build'])}")

    h("Evolution 3: frozen dataclass — immutable, hashable, cacheable")
    @dataclass(frozen=True)
    class BuildKey:
        """Immutable cache key — safe as dict key, safe in sets."""
        repo:   str
        commit: str
        branch: str

    k1 = BuildKey("org/frontend", "abc123", "main")
    k2 = BuildKey("org/frontend", "abc123", "main")
    result_cache: dict[BuildKey, PipelineResult] = {}
    result_cache[k1] = r   # frozen dataclass IS hashable
    assert result_cache[k2] is r    # same key → cache hit!
    print(f"Frozen key used in dict: {result_cache[k1].status}")

    try:
        k1.commit = "xyz"   # TypeError — frozen!
    except (TypeError, AttributeError) as e:
        print(f"Immutability: {e}")

    h("Evolution 4: dataclass with __post_init__ validation")
    from dataclasses import field

    @dataclass
    class ValidatedPipelineResult:
        pipeline_id:  str
        status:       str
        duration_ms:  int
        steps_run:    list[str] = field(default_factory=list)

        def __post_init__(self):
            if self.status not in ("success", "failure", "cancelled", "timeout"):
                raise ValueError(f"Invalid status: {self.status!r}")
            if self.duration_ms < 0:
                raise ValueError("duration_ms cannot be negative")

    try:
        ValidatedPipelineResult("p1", "WINNER", 100)
    except ValueError as e:
        print(f"__post_init__ validation: {e}")

    h("WHERE DATACLASSES ARE USED IN REAL FRAMEWORKS")
    print("""
  FastAPI: Pydantic models ARE essentially dataclasses with runtime validation.
    Pydantic v2 uses __slots__ and compiled validators under the hood.

  FastAPI with pure dataclasses:
    @app.post("/builds")
    async def create_build(config: PipelineConfig) -> PipelineResult:
        # FastAPI will json-serialize the dataclass automatically

  SQLAlchemy 2.0+: @mapped_class uses dataclass-style field definitions
    class Pipeline(Base):
        __tablename__ = "pipelines"
        id: Mapped[int] = mapped_column(primary_key=True)
        name: Mapped[str] = mapped_column(String(100))

  Python stdlib: dataclass(slots=True) in 3.10+ removes __dict__
    for ~50% memory reduction — important for high-frequency objects
    like BuildFast's 10M+ build records in memory
""")


def main() -> None:
    print("="*70)
    print("PYTHON OOP — Real-World Scenarios (BuildFast CI/CD Platform)")
    print("="*70)
    scenario_encapsulation()
    scenario_inheritance_mro()
    scenario_protocol_vs_abc()
    scenario_descriptors()
    scenario_dataclasses()
    print("\n" + "="*70)
    print("Python OOP scenarios complete ✔")


if __name__ == "__main__":
    if hasattr(sys.stdout, "reconfigure"):
        sys.stdout.reconfigure(encoding="utf-8")
    main()